In [67]:
import pandas as pd

df = pd.read_csv("data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
print("stumpings" in df.columns)
print("role" in df.columns)

(27909, 64)
True
True


/var/folders/3b/khlc6jhj47qcw7sj4kwtxp5m0000gn/T/ipykernel_1179/3975629031.py:3: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/processed/player_match_features.csv")


In [68]:
print("stumpings" in df.columns)
print(df.columns.tolist())

True
['match_id', 'player', 'team', 'opposition', 'venue', 'city', 'date', 'season', 'toss_winner', 'toss_decision', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'batting_position', 'wickets', 'runs_conceded', 'balls_bowled', 'maidens', 'economy', 'total_wickets', 'player_innings', 'team_total', 'total_fantasy_points', 'stumpings', 'batting_position_bucket', 'Total_career_runs', 'Total_career_wickets', 'rolling_avg_fantasy_5', 'rolling_std_fantasy_5', 'rolling_avg_fantasy_3', 'rolling_std_fantasy_10', 'rolling_avg_fantasy_10', 'rolling_avg_runs_5', 'rolling_avg_wickets_5', 'matches_played', 'expanding_season_fantasy_avg', 'expanding_season_fantasy_std', 'batting_contribution', 'bowling_contribution', 'rolling_batting_contribution_5', 'rolling_bowling_contribution_5', 'is_home', 'venue_avg_fantasy', 'venue_std_fantasy', 'career_strike_rate', 'career_economy', 'career_batting_avg', 'role', 'role_encoded', 'opposition_avg_fantasy', 'opposition_std_fantasy', 'venue_avg_innings1'

In [69]:

print(df.columns.tolist())
print(df.shape)

['match_id', 'player', 'team', 'opposition', 'venue', 'city', 'date', 'season', 'toss_winner', 'toss_decision', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'batting_position', 'wickets', 'runs_conceded', 'balls_bowled', 'maidens', 'economy', 'total_wickets', 'player_innings', 'team_total', 'total_fantasy_points', 'stumpings', 'batting_position_bucket', 'Total_career_runs', 'Total_career_wickets', 'rolling_avg_fantasy_5', 'rolling_std_fantasy_5', 'rolling_avg_fantasy_3', 'rolling_std_fantasy_10', 'rolling_avg_fantasy_10', 'rolling_avg_runs_5', 'rolling_avg_wickets_5', 'matches_played', 'expanding_season_fantasy_avg', 'expanding_season_fantasy_std', 'batting_contribution', 'bowling_contribution', 'rolling_batting_contribution_5', 'rolling_bowling_contribution_5', 'is_home', 'venue_avg_fantasy', 'venue_std_fantasy', 'career_strike_rate', 'career_economy', 'career_batting_avg', 'role', 'role_encoded', 'opposition_avg_fantasy', 'opposition_std_fantasy', 'venue_avg_innings1', 've

In [70]:
career_stumpings = df.groupby("player")["stumpings"].sum().reset_index()
career_stumpings.columns = ["player", "career_stumpings"]

career_stats = career_stats.drop(columns=["career_stumpings"], errors="ignore")
career_stats = career_stats.merge(career_stumpings, on="player", how="left")

In [71]:
print(career_stats.columns.tolist())

['player', 'total_matches', 'career_runs', 'career_balls_bowled', 'career_wickets', 'career_balls_faced', 'career_runs_conceded', 'avg_bat_position', 'career_strike_rate', 'career_economy', 'career_batting_avg', 'role', 'career_stumpings']


In [72]:
role_lookup = df[["player", "role"]].drop_duplicates(subset="player")
career_stats = career_stats.merge(role_lookup, on="player", how="left")
print(career_stats["role"].value_counts())

KeyError: 'role'

In [63]:

# 1. role-specific raw score
def compute_raw_score(row):
    if row["role"] == "batter":
        return (row["career_batting_avg"] * 1.0) + (row["career_strike_rate"] * 0.15)
    elif row["role"] == "bowler":
        return (row["career_wickets_per_match"] * 25) - (row["career_economy"] * 1.5)
    elif row["role"] == "allrounder":
        return (
            (row["career_batting_avg"] * 0.6)
            + (row["career_strike_rate"] * 0.08)
            + (row["career_wickets_per_match"] * 15)
            - (row["career_economy"] * 1.0)
        )
    else:
        return 5.0

career_stats["raw_score"] = career_stats.apply(compute_raw_score, axis=1)

# 2. normalize career score within each role, safely
def safe_normalize(x):
    if x.max() == x.min():
        return pd.Series(0.5, index=x.index)
    return (x - x.min()) / (x.max() - x.min())

career_stats["career_score_norm"] = career_stats.groupby("role")["raw_score"].transform(safe_normalize)

# 3. recent form blend
career_stats["recent_form_blend"] = (
    0.6 * career_stats["recent_form_5"].fillna(career_stats["recent_form_10"])
    + 0.4 * career_stats["recent_form_10"].fillna(career_stats["recent_form_5"])
).fillna(0)

career_stats["recent_form_norm"] = career_stats.groupby("role")["recent_form_blend"].transform(safe_normalize)

career_stats["final_score_norm"] = (
    0.5 * career_stats["career_score_norm"] + 0.5 * career_stats["recent_form_norm"]
)


print(career_stats["career_score_norm"].min(), career_stats["career_score_norm"].max())
print(career_stats["final_score_norm"].min(), career_stats["final_score_norm"].max())
print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])]
      [["player", "role", "career_score_norm", "recent_form_norm", "final_score_norm"]])

KeyError: 'career_wickets_per_match'

In [34]:
career_stats["credit_value"] = 7.0 + (career_stats["final_score_norm"] * (10.5 - 7.0))
career_stats["credit_value"] = career_stats["credit_value"].round(1)

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])]
      [["player", "role", "final_score_norm", "credit_value"]])

print(career_stats["credit_value"].describe())

        player        role  final_score_norm  credit_value
250  HH Pandya  allrounder          0.429253           8.5
295  JJ Bumrah      bowler          0.412255           8.4
761    V Kohli      batter          0.699558           9.4
count    811.000000
mean       8.390752
std        0.509055
min        7.000000
25%        8.100000
50%        8.400000
75%        8.700000
max       10.400000
Name: credit_value, dtype: float64


In [35]:
career_stats.to_csv("data/processed/player_credits.csv", index=False)
print("Saved player credits")

Saved player credits
